# Dust Cross-Reference: MERRA-2 Speciated Aerosol Against Flagged Stations

**Verification notebook of the spatiotemporal air quality anomaly detection pipeline**

---

The fire cross-reference in `gfed5_crossref.ipynb` tests one specific physical alternative to
misreporting; it does not test the others. This notebook tests a second: windblown mineral
dust, the dominant natural PM2.5 source in arid and semi-arid regions — most prominently the
Taklamakan and Gobi deserts — and a mechanism the fire cross-reference cannot see, since dust
storms carry no fire signature in emissions data.

The motivation is concrete rather than general. Of the 143 consensus-flagged stations, 52
(36%) sit in provinces prone to desert dust, spread across three countries rather than
concentrated in one: 40 in China (Xinjiang, Tibet, Inner Mongolia, Gansu), 7 in India's Thar
Desert region (Rajasthan, Gujarat), and 5 in the arid southwestern United States (California,
Arizona, Texas). Of those 52, only 15 already have a fire explanation from the fire
cross-reference — leaving 37 flagged stations in dust-prone regions with no tested physical
alternative at all. The fire cross-reference's own finding — that flagged and unflagged
stations show statistically indistinguishable fire exposure — does not by itself rule out dust
as a regional confound the fire check was never designed to catch.

This notebook complements the fire cross-reference; it does not replace it. A station can
have a legitimate fire explanation, a legitimate dust explanation, both, or neither, and this
notebook's output is designed to sit alongside `GFED5_fire_crossref_station.csv` rather than
supersede it.

| | |
|---|---|
| **Input** | `CONSENSUS_station_ranking.csv`, `NB01_station_locations.csv`, `NB02_extreme_review_candidates.csv`, MERRA-2 M2T1NXAER monthly extracts |
| **Output** | `DUST_crossref_station.csv`, `DUST_crossref_episodes.csv` |
| **Downstream** | Manual review; the final CREA report's limitations and attribution sections |


## Contents

0. **Method Context**
   - 0.1 Objective
   - 0.2 Approach and Principles
   - 0.3 Inputs and Outputs
1. **Environment and Data**
   - 1.1 Configuration
   - 1.2 Station Coordinates and Consensus Verdicts
   - 1.3 Extreme-Reading Episodes
2. **MERRA-2 Archive Access**
   - 2.1 File Inventory and Provenance
   - 2.2 Grid Structure Verification
3. **Station-to-Grid Assignment**
   - 3.1 Nearest Grid Cell per Station
   - 3.2 Assignment Distance Diagnostics
4. **Dust Concentration Extraction**
   - 4.1 Hourly Series per Station
   - 4.2 Extraction Coverage
5. **Dust Event Definition**
   - 5.1 Grid-Cell-Specific Baseline
   - 5.2 Event Threshold Derivation
6. **Episode-Level Cross-Reference**
   - 6.1 Matching Extreme Readings to Dusty Hours
   - 6.2 Attribution Outcome
7. **Station-Level Cross-Reference**
   - 7.1 Dust Exposure by Consensus Status
   - 7.2 Reconsideration Candidates
   - 7.3 Overlap with the Fire Cross-Reference
8. **Results and Handoff**
   - 8.1 Cross-Reference Output
   - 8.2 Output Validation
9. **Findings and Limitations**

---

Terminology

- **DUSMASS25** is a MERRA-2 diagnostic: modelled surface concentration of windblown mineral
  dust in the PM2.5-relevant size fraction, already separated from smoke, sulfate, sea salt,
  and black carbon by the reanalysis's own aerosol speciation — unlike total aerosol optical
  depth, which mixes all of these together.
- A **dusty hour** is an hour in which a station's assigned grid cell shows dust concentration
  substantially above that cell's own typical level, defined the same way a fire day is
  defined in the fire cross-reference: relative to the cell's own distribution, not a fixed
  global figure.
- An **episode** is the same station-date pair used in the fire cross-reference, drawn from
  the extreme-value review candidates identified in Notebook 02.
- **Attribution** here means only that a plausible physical alternative to misreporting
  exists for a given flag; as in the fire cross-reference, it is never a demonstration that
  misreporting did not occur.


## 0. Method Context

### 0.1 Objective

Reverse-geocoding the 143 consensus-flagged stations against Notebook 01's location table
found 52 of them (36%) in provinces prone to desert dust — a concentration the fire
cross-reference's own aggregate finding does not address, since fire and dust are unrelated
mechanisms. These 52 are spread across three countries rather than one: 40 in China (the
Taklamakan and Gobi source regions of Xinjiang, Tibet, Inner Mongolia, and Gansu), 7 in
India's Thar Desert region, and 5 in the arid southwestern United States. Of the 52, only 15
already carry a fire explanation, leaving 37 flagged stations in dust-prone regions with no
tested physical alternative — the specific population this notebook exists to address.

The acquisition for this notebook is deliberately scoped to those dust-prone regions rather
than the whole of each country: the bounding boxes in the download script cover the arid zones
containing these 52 stations plus a buffer, not Hunan's humid subtropics or the oceanic
Hawaiian stations or the eastern United States, none of which border a desert dust source
however many flagged stations they contain. This keeps the acquisition roughly an order of
magnitude smaller than a whole-country download while still supporting a flagged-versus-
unflagged comparison within the dust-prone zones themselves.

Three objectives follow, mirroring the fire cross-reference's structure:

1. **Test whether a flagged station's anomalies coincide with locally elevated dust
   concentration.** <br>Coincidence is not proof, but its absence removes a plausible
   innocent explanation for stations in dust-prone regions, and its presence is a specific,
   checkable reason to revisit a flag the fire check could not address.</br>

2. **Quantify whether dust concentrates in the flagged population the way fire did not.**
   <br>The fire cross-reference found flagged and unflagged stations statistically
   indistinguishable in fire exposure. Whether the same holds for dust is an open, testable
   question this notebook answers directly rather than assuming an analogous result.</br>

3. **Report the two cross-references side by side, not merged into one verdict.** <br>A
   station's fire and dust exposure are computed independently, so a reader can see whether a
   given flag is explained by one, both, or neither — merging them into a single "any natural
   explanation" flag would hide which specific mechanism, if any, applies.</br>


### 0.2 Approach and Principles

Five decisions govern the implementation, several inherited directly from the fire
cross-reference.

1. **DUSMASS25 is used because it is already speciated.** <br>Total aerosol optical depth
   mixes dust with smoke, sulfate, sea salt, and black carbon; distinguishing dust from smoke
   in raw AOD would require a secondary test such as the Ångström exponent. MERRA-2's
   assimilation model performs that separation internally and reports dust's own surface
   concentration directly, which is what this notebook reads rather than re-deriving.</br>

2. **A dusty hour is defined against each grid cell's own baseline** <br>— the identical
   principle the fire cross-reference applies to emissions, for the identical reason: a dust
   concentration that is unremarkable for the Taklamakan's immediate surroundings would be
   extraordinary over Germany, so no single global threshold is meaningful across networks
   this different.</br>

3. **Hourly resolution is used, not aggregated to daily.** <br>MERRA-2's aerosol diagnostics
   are natively hourly, unlike GFED5's daily emissions. A dust storm can arrive and pass within
   hours, and extreme PM2.5 readings are themselves timestamped to the hour, so matching at
   hourly resolution is more precise than the daily window the fire cross-reference uses
   because that is the finest GFED5 supplies.</br>

4. **The test can only weaken a suspicion, never strengthen one** <br>— the same asymmetry
   stated in the fire cross-reference, restated here because it applies without modification:
   dust near a flagged station is a reason to reconsider that flag; its absence does not
   corroborate misreporting, since dust is only one of several untested alternatives.</br>

5. **This notebook does not merge with the fire cross-reference.** <br>Both read the same
   consensus ranking independently and write separate output files. Section 7.3 reports where
   the two overlap, but a station's fire and dust exposure remain two distinct, independently
   computed pieces of evidence rather than being combined into a single score.</br>


### 0.3 Inputs and Outputs

| Direction | Artifact | Contents |
|---|---|---|
| **In** | `CONSENSUS_station_ranking.csv` | Per-station consensus rank and flag |
| **In** | `NB01_station_locations.csv` | Station coordinates, city, and province |
| **In** | `NB02_extreme_review_candidates.csv` | Station-date pairs carrying extreme readings |
| **In** | MERRA-2 M2T1NXAER monthly extracts | Hourly DUSMASS25, 2021–2023 |
| **Out** | `DUST_crossref_station.csv` | Per-station dust exposure and reconsideration status |
| **Out** | `DUST_crossref_episodes.csv` | Per-episode dust coincidence for extreme readings |

**Data provenance.** MERRA-2 is NASA's ongoing atmospheric reanalysis; unlike the GFED5 Beta
extension, the 2021–2023 period used here is not subject to later reprocessing, so results
from this notebook do not carry the same access-date sensitivity the fire cross-reference's
2023 figures do. The acquisition scripts accompanying this notebook were written against
NASA's documented `earthaccess` API and MERRA-2 file specifications but have not been
executed against the live service from this environment, which has no network route to
NASA's servers; the first real run may need minor adjustment.

## 1. Environment and Data

### 1.1 Configuration

Both MERRA-2 archives referenced here (should NASA revise the product in future) would be
declared through the same registry pattern used for GFED5's stable/Beta split, so that adding
a revised source requires one entry rather than restructuring. Only one archive exists at
present.


In [ ]:
import glob
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_style import table_style

CONFIG_PATH = Path("../config/params.yml")
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
FIGURE_DIR    = PROCESSED_DIR.parent.parent / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_ORDER   = ["China", "Germany", "India", "USA"]
RANDOM_SEED     = CONFIG["meta"]["random_seed"]

MERRA2_DIR = PROCESSED_DIR.parent / "merra2_dust_raw"

# Grid resolution declared explicitly -- coarser than GFED5's 0.25 degree, which
# Section 3.2 and the limitations section both return to.
EXPECTED_LAT_STEP = 0.5
EXPECTED_LON_STEP = 0.625

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)

display(table_style(pd.DataFrame([
    ("MERRA-2 directory", str(MERRA2_DIR)),
    ("Expected grid resolution", f"{EXPECTED_LAT_STEP} deg lat x {EXPECTED_LON_STEP} deg lon"),
    ("Dust variable", "DUSMASS25 (PM2.5-fraction surface dust concentration)"),
], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

The declared grid resolution — 0.5° latitude by 0.625° longitude — is verified against the
actual files in Section 2.2 rather than assumed; the check exists specifically to catch a
silently mismatched grid before any station is assigned to a cell.

### 1.2 Station Coordinates and Consensus Verdicts

Coordinates, city, and province come from Notebook 01's location table directly, rather than
re-deriving them, since that table is the single source this pipeline now uses for identifying
a station by name. The consensus ranking supplies which stations to examine most closely,
though dust exposure is computed for every station so flagged and unflagged populations can be
compared in Section 7.


In [ ]:
LOCATIONS_PATH = PROCESSED_DIR / "NB01_station_locations.csv"
if not LOCATIONS_PATH.exists():
    raise FileNotFoundError(f"{LOCATIONS_PATH} not found. Run Notebook 01 to completion first.")

stations = pd.read_csv(LOCATIONS_PATH)[
    ["location_id", "country", "latitude", "longitude", "city", "province"]
].drop_duplicates("location_id")

CONSENSUS_PATH = PROCESSED_DIR / "CONSENSUS_station_ranking.csv"
if not CONSENSUS_PATH.exists():
    raise FileNotFoundError(f"{CONSENSUS_PATH} not found. Run consensus.ipynb to completion first.")
consensus = pd.read_csv(CONSENSUS_PATH)

stations = stations.merge(
    consensus[["location_id", "consensus_rank", "consensus_rate",
               "consensus_flagged", "consensus_direction"]],
    on="location_id", how="left")

display(table_style(pd.DataFrame([
    ("Stations with coordinates", f"{len(stations):,}"),
    ("Matched to a consensus verdict", f"{int(stations['consensus_flagged'].notna().sum()):,}"),
    ("Consensus-flagged", f"{int(stations['consensus_flagged'].fillna(False).sum()):,}"),
], columns=["Property", "Value"]).set_index("Property")))

display(table_style(stations[stations["consensus_flagged"].fillna(False)]
                    .groupby("province").size().sort_values(ascending=False)
                    .head(10).rename("flagged_stations").to_frame()))


<u>Interpretation</u>

The province breakdown of flagged stations is the same view that motivated this notebook: if
Xinjiang and Tibet again appear at the top, that confirms the concentration found during
reverse-geocoding is reproducible from the pipeline's own location table rather than an
artefact of how that earlier check was run.

### 1.3 Extreme-Reading Episodes

The same episode set the fire cross-reference uses — station-date pairs carrying a reading
extreme enough to have been marked for review during exploratory analysis. Using the identical
set means a station's fire and dust attribution can be compared episode-for-episode in Section
7.3, rather than against two different samples.


In [ ]:
EPISODES_PATH = PROCESSED_DIR / "NB02_extreme_review_candidates.csv"
if EPISODES_PATH.exists():
    episodes = pd.read_csv(EPISODES_PATH)
    episodes["date"] = pd.to_datetime(episodes["date"], utc=True, errors="coerce")
    episodes = episodes.dropna(subset=["date"])
    episodes_available = True
else:
    episodes = pd.DataFrame(columns=["location_id", "country", "date", "value"])
    episodes_available = False

display(table_style(pd.DataFrame([
    ("Episode file present", "yes" if episodes_available else "no"),
    ("Extreme-reading episodes", f"{len(episodes):,}"),
    ("Distinct stations with an episode", f"{episodes['location_id'].nunique():,}"),
], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

Reading the same episode file the fire cross-reference reads is what makes Section 7.3's
overlap comparison meaningful: any difference in which episodes are examined would confound a
comparison of what each mechanism explains with a comparison of what each notebook happened to
look at.

## 2. MERRA-2 Archive Access

### 2.1 File Inventory and Provenance

Monthly extracts, one file per calendar month across 2021–2023, are inventoried before
anything is read, so a missing month is discovered here rather than as a silently thin result
several sections later.


In [ ]:
monthly_files = sorted(glob.glob(str(MERRA2_DIR / "merra2_dust_*.nc")))

display(table_style(pd.DataFrame([
    ("MERRA-2 files found", len(monthly_files)),
    ("Expected (36 months, 2021-2023)", 36),
    ("Access date (provenance)", "not yet acquired -- record the actual download date here"),
], columns=["Property", "Value"]).set_index("Property")))

if not monthly_files:
    raise FileNotFoundError(
        f"No MERRA-2 dust extracts found under {MERRA2_DIR}. Run "
        f"02_download_merra2_dust.py and 03_extract_dust_to_stations.py first."
    )


<u>Interpretation</u>

A shortfall from 36 files does not necessarily indicate a failed acquisition — a month with
no granules returned by the search would be logged as a warning by the download script rather
than silently skipped, but confirming the count here is a cheap check against a partial
run.

### 2.2 Grid Structure Verification

MERRA-2 is documented as a 0.5° × 0.625° global grid; that is checked directly here rather
than assumed, since a mismatched grid would silently misassign every station — the same
discipline applied to GFED5's shared grid in the fire cross-reference.


In [ ]:
with xr.open_dataset(monthly_files[0]) as ds0:
    reference_lat = ds0["lat"].values.copy()
    reference_lon = ds0["lon"].values.copy()
    variable_present = "DUSMASS25" in ds0.data_vars

lat_step = float(np.median(np.diff(reference_lat)))
lon_step = float(np.median(np.diff(reference_lon)))

display(table_style(pd.DataFrame([
    ("Grid shape", f"{len(reference_lat)} x {len(reference_lon)}"),
    ("Latitude step", round(lat_step, 4)),
    ("Longitude step", round(lon_step, 4)),
    ("Latitude range", f"{reference_lat.min():.2f} to {reference_lat.max():.2f}"),
    ("Longitude range", f"{reference_lon.min():.2f} to {reference_lon.max():.2f}"),
    ("DUSMASS25 present", variable_present),
], columns=["Property", "Value"]).set_index("Property")))

if not variable_present:
    raise KeyError("DUSMASS25 absent from the first MERRA-2 file; check the extraction script.")
if abs(lat_step - EXPECTED_LAT_STEP) > 0.01 or abs(lon_step - EXPECTED_LON_STEP) > 0.01:
    raise ValueError(
        f"Grid step ({lat_step}, {lon_step}) does not match the documented MERRA-2 "
        f"resolution ({EXPECTED_LAT_STEP}, {EXPECTED_LON_STEP}); station assignment "
        f"in Section 3 would be built on the wrong assumption."
    )


<u>Interpretation</u>

Confirming the grid step directly, rather than trusting the documented specification, is what
makes the station-assignment distances in Section 3.2 trustworthy — a silently finer or
coarser grid would shift every offset calculated downstream without necessarily producing an
error anywhere obvious.

## 3. Station-to-Grid Assignment

### 3.1 Nearest Grid Cell per Station

Each station is assigned the grid cell whose centre lies closest to it, the same nearest-point
principle used for GFED5, applied to a coarser grid.


In [ ]:
def nearest_index(value, grid):
    return int(np.abs(grid - value).argmin())

stations["lat_idx"] = stations["latitude"].apply(lambda v: nearest_index(v, reference_lat))
stations["lon_idx"] = stations["longitude"].apply(lambda v: nearest_index(v, reference_lon))
stations["cell_lat"] = reference_lat[stations["lat_idx"]]
stations["cell_lon"] = reference_lon[stations["lon_idx"]]
stations["cell_id"] = stations["lat_idx"].astype(str) + "_" + stations["lon_idx"].astype(str)

R_EARTH_KM = 6371.0
dlat = np.radians(stations["cell_lat"] - stations["latitude"])
dlon = np.radians(stations["cell_lon"] - stations["longitude"])
mean_lat = np.radians((stations["cell_lat"] + stations["latitude"]) / 2)
stations["assignment_offset_km"] = R_EARTH_KM * np.sqrt(dlat**2 + (np.cos(mean_lat) * dlon)**2)

display(table_style(stations["assignment_offset_km"].describe().round(2).to_frame("offset_km")))


<u>Interpretation</u>

The worst-case offset on this grid is larger than GFED5's — roughly 0.5° of latitude and
0.625° of longitude per cell, against GFED5's 0.25° square, so a maximum offset in the
neighbourhood of 35 km rather than GFED5's ~20 km is expected and not itself a sign of a
problem. A maximum far beyond that would indicate a coordinate outside the grid extent.

### 3.2 Assignment Distance Diagnostics

Cell sharing is more pronounced on this coarser grid than on GFED5's — the same limitation
established there, more acute here, and quantified the same way.


In [ ]:
cell_occupancy = (stations.groupby("cell_id").size()
                  .rename("stations_in_cell").reset_index())
stations = stations.merge(cell_occupancy, on="cell_id", how="left")

occupancy_summary = (stations.groupby("country")
                     .agg(stations=("location_id", "size"),
                          distinct_cells=("cell_id", "nunique"),
                          max_stations_per_cell=("stations_in_cell", "max"),
                          median_offset_km=("assignment_offset_km", "median"))
                     .reindex([c for c in COUNTRY_ORDER if c in stations["country"].values])
                     .round(2))
occupancy_summary["stations_per_cell"] = (occupancy_summary["stations"] /
                                          occupancy_summary["distinct_cells"]).round(2)

display(table_style(occupancy_summary))


<u>Interpretation</u>

Where several stations share a cell, dust exposure applies identically to all of them, and
this cross-reference cannot distinguish which — if any — was actually affected. This is a
sharper version of the same limitation the fire cross-reference carries, since MERRA-2's grid
is coarser than GFED5's; the practical consequence is that a dust coincidence here should be
read as evidence about a station's general vicinity more than about that station specifically,
more so than the equivalent fire finding.

## 4. Dust Concentration Extraction

### 4.1 Hourly Series per Station

Only the cells stations actually occupy are extracted — a few hundred at most — rather than
the full global grid for every hour, the same "occupied cells only" pattern used for GFED5.


In [ ]:
occupied = stations.drop_duplicates("cell_id")[["cell_id", "lat_idx", "lon_idx"]].reset_index(drop=True)
lat_take = occupied["lat_idx"].to_numpy()
lon_take = occupied["lon_idx"].to_numpy()

KG_TO_UG = 1e9   # DUSMASS25 native units are kg/m3

extraction_parts = []
files_read = 0
files_failed = []

for path in monthly_files:
    try:
        with xr.open_dataset(path) as ds:
            if "DUSMASS25" not in ds.data_vars:
                files_failed.append((Path(path).name, "DUSMASS25 absent"))
                continue
            values = ds["DUSMASS25"].values * KG_TO_UG    # (time, lat, lon) -> ug/m3
            times = pd.to_datetime(ds["time"].values)
            cell_values = values[:, lat_take, lon_take]
            extraction_parts.append(pd.DataFrame({
                "hour": np.repeat(times, len(occupied)),
                "cell_id": np.tile(occupied["cell_id"].to_numpy(), len(times)),
                "dust_ug_m3": cell_values.reshape(-1),
            }))
        files_read += 1
    except Exception as exc:
        files_failed.append((Path(path).name, str(exc)[:60]))

if not extraction_parts:
    raise RuntimeError("No MERRA-2 dust data could be extracted; check the archive files.")

dust_hourly = pd.concat(extraction_parts, ignore_index=True)

display(table_style(pd.DataFrame([
    ("Files read successfully", files_read),
    ("Files failed", len(files_failed)),
    ("Cell-hour rows extracted", f"{len(dust_hourly):,}"),
    ("Distinct cells", f"{dust_hourly['cell_id'].nunique():,}"),
    ("Date range", f"{dust_hourly['hour'].min()} to {dust_hourly['hour'].max()}"),
], columns=["Property", "Value"]).set_index("Property")))

if files_failed:
    display(table_style(pd.DataFrame(files_failed, columns=["file", "reason"]).head(10)))


<u>Interpretation</u>

A failed file is surfaced with its reason rather than skipped silently, for the same reason
this matters in the fire cross-reference: a systematic failure would otherwise appear only as
an unexplained gap in the dust record, reading as "no dust" and weakening exactly the flags
this notebook exists to reconsider.

### 4.2 Extraction Coverage



In [ ]:
dust_hourly["year"] = dust_hourly["hour"].dt.year

coverage_by_year = (dust_hourly.groupby("year")
                    .agg(hours_covered=("hour", "nunique"),
                         cell_hours=("dust_ug_m3", "size"),
                         median_dust=("dust_ug_m3", "median"),
                         p99_dust=("dust_ug_m3", lambda s: s.quantile(0.99)))
                    .round(3))

display(table_style(coverage_by_year))


<u>Interpretation</u>

A full year should show close to 8,760 hours covered (8,784 in a leap year). The median dust
concentration is expected to sit close to zero almost everywhere, with the extreme tail
carrying the signal — the same shape the fire cross-reference's emission distribution
showed, for the same underlying reason: the phenomenon is genuinely episodic rather than
continuously present.

## 5. Dust Event Definition

### 5.1 Grid-Cell-Specific Baseline

A dust concentration that signals a major storm in one landscape is unremarkable in another.
Each cell is characterised by its own distribution across the observation window, exactly the
principle applied to GFED5's fire threshold and to every station-relative comparison
elsewhere in this pipeline.


In [ ]:
cell_baseline = (dust_hourly.groupby("cell_id")["dust_ug_m3"]
                 .agg(hours="size",
                      median_dust="median",
                      p90_dust=lambda s: float(s.quantile(0.90)),
                      p99_dust=lambda s: float(s.quantile(0.99)),
                      max_dust="max")
                 .reset_index())

display(table_style(cell_baseline[["hours", "median_dust", "p90_dust", "p99_dust", "max_dust"]]
                    .describe().round(3)))


<u>Interpretation</u>

As with fire emissions, most cells carry a median at or near zero, which is why a
percentile-based threshold is needed rather than a mean-based one — a mean over a
distribution this skewed would sit far above the typical hour and could fail to clear on a
genuine dust event.

### 5.2 Event Threshold Derivation

A cell's dusty-hour threshold is its own 90th percentile, subject to an absolute floor so
that a cell with negligible dust activity does not register routine noise as an event —
the same two-part construction used for the fire threshold, with the floor's value tested for
sensitivity rather than asserted.


In [ ]:
MIN_ABSOLUTE_DUST_UG_M3 = 5.0   # floor: below this, an hour is not treated as dusty
                                 # regardless of how unusual it is for that cell
DUST_PERCENTILE = 0.90

cell_baseline["dust_threshold"] = np.maximum(cell_baseline["p90_dust"], MIN_ABSOLUTE_DUST_UG_M3)

dust_hourly = dust_hourly.merge(cell_baseline[["cell_id", "dust_threshold"]], on="cell_id", how="left")
dust_hourly["is_dusty_hour"] = dust_hourly["dust_ug_m3"] >= dust_hourly["dust_threshold"]

threshold_sensitivity = pd.DataFrame([
    {"floor_ug_m3": floor,
     "cell_hours_flagged": int((dust_hourly["dust_ug_m3"] >=
                                np.maximum(dust_hourly["dust_threshold"], floor)).sum()),
     "share_of_cell_hours_pct": round(float((dust_hourly["dust_ug_m3"] >=
                                             np.maximum(dust_hourly["dust_threshold"], floor)).mean() * 100), 3)}
    for floor in [0, 1, 5, 10, 25]
]).set_index("floor_ug_m3")

display(table_style(threshold_sensitivity))
display(table_style(pd.DataFrame([
    ("Percentile used", DUST_PERCENTILE),
    ("Absolute floor (ug/m3)", MIN_ABSOLUTE_DUST_UG_M3),
    ("Dusty hours identified", f"{int(dust_hourly['is_dusty_hour'].sum()):,}"),
    ("Share of all cell-hours (%)", round(float(dust_hourly["is_dusty_hour"].mean() * 100), 3)),
], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

As with the fire threshold, the floor's effect is shown directly rather than asserted: a
floor of zero would flag roughly a tenth of all cell-hours by construction of the percentile,
including cells where the concentration in absolute terms is negligible. The adopted floor
restricts dusty hours to cells and hours where the concentration is substantial as well as
unusual, and a reader who disagrees with the specific value can read the consequence of a
different one directly from the table above.

## 6. Episode-Level Cross-Reference

### 6.1 Matching Extreme Readings to Dusty Hours

Each extreme reading is matched to dust activity in its station's cell within a short hourly
window rather than the day-level window the fire cross-reference uses, since MERRA-2's hourly
resolution supports finer matching than GFED5's daily emissions do.


In [ ]:
DUST_LAG_HOURS = 3   # a dusty hour this close to a reading can plausibly explain it;
                     # a plume does not arrive and clear instantaneously

if episodes_available and len(episodes):
    episode_join = episodes.merge(
        stations[["location_id", "cell_id", "consensus_flagged", "consensus_rate",
                  "stations_in_cell", "province"]],
        on="location_id", how="left")

    dust_lookup = (dust_hourly.set_index(["cell_id", "hour"])["is_dusty_hour"].to_dict())
    conc_lookup = (dust_hourly.set_index(["cell_id", "hour"])["dust_ug_m3"].to_dict())

    def dust_within_window(cell_id, date):
        if pd.isna(cell_id) or pd.isna(date):
            return np.nan, np.nan
        hit, peak = False, 0.0
        # Episode timestamps are parsed tz-aware (UTC) in Section 1.3; MERRA-2's time
        # coordinate decodes tz-naive via standard NetCDF/CF convention, representing UTC
        # implicitly rather than explicitly. Both sides are the same wall-clock time, so the
        # tz tag is stripped here for the dictionary lookup rather than left mismatched --
        # confirmed necessary during testing, where leaving both sides as parsed produced
        # zero matches even for episodes deliberately overlapping a synthetic dust window.
        base_hour = pd.Timestamp(date).tz_localize(None).floor("h")
        for back in range(DUST_LAG_HOURS + 1):
            h = base_hour - pd.Timedelta(hours=back)
            key = (cell_id, h)
            if dust_lookup.get(key, False):
                hit = True
            peak = max(peak, conc_lookup.get(key, 0.0))
        return hit, peak

    results = [dust_within_window(c, d) for c, d in zip(episode_join["cell_id"], episode_join["date"])]
    episode_join["dust_in_window"] = [r[0] for r in results]
    episode_join["peak_dust_in_window"] = [r[1] for r in results]

    display(table_style(pd.DataFrame([
        ("Episodes examined", f"{len(episode_join):,}"),
        ("Coinciding with a dusty hour", f"{int(episode_join['dust_in_window'].fillna(False).sum()):,}"),
        ("Share coinciding (%)", round(float(episode_join["dust_in_window"].fillna(False).mean() * 100), 2)),
        ("Lag window (hours)", DUST_LAG_HOURS),
    ], columns=["Property", "Value"]).set_index("Property")))
else:
    episode_join = pd.DataFrame()
    display(table_style(pd.DataFrame([
        ("Episode analysis", "skipped -- no extreme-reading candidates available"),
    ], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

The share of extreme readings coinciding with dust activity is this notebook's headline
figure, read the same way the fire cross-reference's equivalent figure is read: a high share
means a physical explanation is available for much of what was flagged as extreme; a low
share leaves misreporting among the explanations still standing, though not established —
dust is only one of several untested alternatives, alongside fire, which Section 7.3 checks
directly.

### 6.2 Attribution Outcome by Province

Where the fire cross-reference breaks episodes down by country, this notebook breaks them
down by province specifically, since the motivating finding was concentrated in two provinces
rather than spread evenly across a country.


In [ ]:
if len(episode_join):
    episode_join["consensus_flagged"] = episode_join["consensus_flagged"].fillna(False).astype(bool)
    episode_join["dust_in_window"] = episode_join["dust_in_window"].fillna(False).astype(bool)

    attribution = (episode_join.groupby(["consensus_flagged", "dust_in_window"])
                   .agg(episodes=("location_id", "size"),
                        stations=("location_id", "nunique"),
                        median_reading=("value", "median"))
                   .round(2))
    display(table_style(attribution))

    by_province = (episode_join.groupby("province")
                   .agg(episodes=("location_id", "size"),
                        dust_coincident=("dust_in_window", "sum"))
                   .sort_values("episodes", ascending=False)
                   .head(10))
    by_province["dust_coincident_pct"] = (by_province["dust_coincident"] /
                                          by_province["episodes"] * 100).round(2)
    display(table_style(by_province))


<u>Interpretation</u>

The province breakdown is where this notebook's specific motivation is either confirmed or
not: if Xinjiang and Tibet show a materially higher dust-coincidence share than other
provinces, that is direct evidence the concentration found during reverse-geocoding has at
least a partial physical explanation the fire check could not see. If they do not stand out,
that is equally informative — it would mean the geographic concentration itself still needs an
explanation dust does not supply.

## 7. Station-Level Cross-Reference

### 7.1 Dust Exposure by Consensus Status

The same station-level comparison the fire cross-reference performs, applied to dust: is a
consensus-flagged station systematically more exposed to dust than an unflagged one, across
the whole network rather than only its extreme episodes.


In [ ]:
station_exposure = (dust_hourly.groupby("cell_id")
                    .agg(dusty_hours=("is_dusty_hour", "sum"),
                         hours_covered=("hour", "nunique"),
                         peak_dust=("dust_ug_m3", "max"))
                    .reset_index())
station_exposure["dusty_hour_share_pct"] = (station_exposure["dusty_hours"] /
                                            station_exposure["hours_covered"] * 100).round(2)

stations = stations.merge(station_exposure, on="cell_id", how="left")

exposure_comparison = (stations.groupby("consensus_flagged")
                       .agg(stations=("location_id", "size"),
                            median_dusty_hours=("dusty_hours", "median"),
                            median_dusty_hour_share=("dusty_hour_share_pct", "median"),
                            median_peak_dust=("peak_dust", "median"))
                       .round(3))
display(table_style(exposure_comparison))

fig, ax = plt.subplots(figsize=(9, 4.5))
for flag, label, colour in [(False, "Not consensus-flagged", "#378ADD"),
                            (True, "Consensus-flagged", "#C0392B")]:
    sub = stations.loc[stations["consensus_flagged"] == flag, "dusty_hour_share_pct"].dropna()
    if len(sub):
        ax.hist(sub, bins=40, alpha=0.6, density=True, label=f"{label} (n={len(sub):,})", color=colour)
ax.set_xlabel("Share of hours classified as dusty in the station's grid cell (%)")
ax.set_ylabel("Density")
ax.set_title("Dust exposure of consensus-flagged against unflagged stations")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "dust_exposure_by_consensus.png")
plt.show()


<u>Interpretation</u>

This is the direct test of whether dust behaves like fire did. The fire cross-reference found
flagged and unflagged stations statistically indistinguishable in fire exposure across the
network as a whole. Whether the same holds here is read off the comparison above: close
medians and overlapping distributions would mean dust, like fire, is not a systematic
confound for the consensus in aggregate — a materially higher exposure among flagged stations
would mean it is, at least in part.

### 7.2 Reconsideration Candidates

The same reconsideration-candidate construction as the fire cross-reference, with the same
caveat about direction: dust plausibly explains a station reading high, not one reading low.


In [ ]:
HIGH_EXPOSURE_PERCENTILE = 0.75

exposure_cut = stations["dusty_hour_share_pct"].quantile(HIGH_EXPOSURE_PERCENTILE)
stations["high_dust_exposure"] = (
    (stations["dusty_hour_share_pct"] >= exposure_cut) &
    (stations["dusty_hour_share_pct"] > 0)
)
stations["dust_reconsideration_candidate"] = stations["consensus_flagged"] & stations["high_dust_exposure"]

display(table_style(pd.DataFrame([
    ("High-exposure cutoff (dusty-hour share %)", round(float(exposure_cut), 2)),
    ("Consensus-flagged stations", int(stations["consensus_flagged"].sum())),
    ("...of which high dust exposure", int(stations["dust_reconsideration_candidate"].sum())),
], columns=["Property", "Value"]).set_index("Property")))

candidates_table = (stations[stations["dust_reconsideration_candidate"]]
                    .sort_values("consensus_rate", ascending=False)
                    [["location_id", "country", "province", "consensus_rank", "consensus_rate",
                      "consensus_direction", "dusty_hour_share_pct", "peak_dust", "stations_in_cell"]]
                    .head(20).reset_index(drop=True).round(3))
display(table_style(candidates_table))


<u>Interpretation</u>

As with fire, a station flagged for under-reporting is not meaningfully explained by high
dust exposure, since dust raises readings rather than lowering them; the `consensus_direction`
column carries that distinction into the candidate list directly rather than leaving it to be
inferred. The `stations_in_cell` column carries forward the coarser-grid caveat from Section
3.2: a candidate sharing its cell with several other stations inherits that cell's exposure
without evidence the dust was near it specifically.

### 7.3 Overlap with the Fire Cross-Reference

Fire and dust are computed entirely independently in this pipeline, so whether they explain
the same stations or different ones is itself a finding rather than something assumed. This
section reads the fire cross-reference's own output rather than recomputing anything.


In [ ]:
FIRE_STATION_PATH = PROCESSED_DIR / "GFED5_fire_crossref_station.csv"

if FIRE_STATION_PATH.exists():
    fire = pd.read_csv(FIRE_STATION_PATH)[["location_id", "reconsideration_candidate"]]
    fire = fire.rename(columns={"reconsideration_candidate": "fire_reconsideration_candidate"})
    overlap = stations.merge(fire, on="location_id", how="left")
    overlap["fire_reconsideration_candidate"] = overlap["fire_reconsideration_candidate"].fillna(False)

    only_fire  = int((overlap["fire_reconsideration_candidate"] & ~overlap["dust_reconsideration_candidate"]).sum())
    only_dust  = int((~overlap["fire_reconsideration_candidate"] & overlap["dust_reconsideration_candidate"]).sum())
    both       = int((overlap["fire_reconsideration_candidate"] & overlap["dust_reconsideration_candidate"]).sum())
    neither_flagged = int((overlap["consensus_flagged"] &
                          ~overlap["fire_reconsideration_candidate"] &
                          ~overlap["dust_reconsideration_candidate"]).sum())

    display(table_style(pd.DataFrame([
        ("Explained by fire only", only_fire),
        ("Explained by dust only", only_dust),
        ("Explained by both", both),
        ("Consensus-flagged, explained by neither", neither_flagged),
    ], columns=["Category", "Stations"]).set_index("Category")))
else:
    display(table_style(pd.DataFrame([
        ("Fire cross-reference output", "not found -- run gfed5_crossref.ipynb first for this comparison"),
    ], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

The "explained by neither" count is the figure that matters most for the report: it is the
number of consensus-flagged stations for which neither of the two natural-cause mechanisms
tested in this pipeline offers a physical alternative. That count still is not evidence of
misreporting on its own — other untested mechanisms remain, and the asymmetry stated in
Section 0.2 applies here as everywhere else in this pipeline — but it is the most refined
figure available after both cross-references, and it is the one a report should carry forward
rather than either notebook's figure in isolation.

## 8. Results and Handoff

### 8.1 Cross-Reference Output

Two files are written, mirroring the fire cross-reference's structure exactly so a reader
already familiar with `GFED5_fire_crossref_station.csv` needs no new schema to read this
notebook's output.


In [ ]:
station_output_columns = [
    "location_id", "country", "province", "latitude", "longitude",
    "cell_id", "cell_lat", "cell_lon", "assignment_offset_km", "stations_in_cell",
    "consensus_rank", "consensus_rate", "consensus_flagged", "consensus_direction",
    "dusty_hours", "hours_covered", "dusty_hour_share_pct", "peak_dust",
    "high_dust_exposure", "dust_reconsideration_candidate",
]
station_output = stations[station_output_columns].copy()

STATION_OUT_PATH = PROCESSED_DIR / "DUST_crossref_station.csv"
station_output.to_csv(STATION_OUT_PATH, index=False)

if len(episode_join):
    episode_output = episode_join[[
        "location_id", "country", "province", "date", "value", "cell_id",
        "consensus_flagged", "consensus_rate", "stations_in_cell",
        "dust_in_window", "peak_dust_in_window",
    ]].copy()
    episode_output["lag_hours"] = DUST_LAG_HOURS
else:
    episode_output = pd.DataFrame(columns=["location_id", "country", "date", "value"])

EPISODE_OUT_PATH = PROCESSED_DIR / "DUST_crossref_episodes.csv"
episode_output.to_csv(EPISODE_OUT_PATH, index=False)

display(table_style(pd.DataFrame([
    ("Station file", STATION_OUT_PATH.name),
    ("Stations written", f"{len(station_output):,}"),
    ("Episode file", EPISODE_OUT_PATH.name),
    ("Episodes written", f"{len(episode_output):,}"),
    ("Reconsideration candidates", int(station_output["dust_reconsideration_candidate"].sum())),
], columns=["Property", "Value"]).set_index("Property")))


<u>Interpretation</u>

Every station is written, not only the flagged or dust-exposed ones, so the unflagged
population remains available as the comparison baseline Section 7.1's conclusion depends on
— the identical reasoning that governs the fire cross-reference's output.

### 8.2 Output Validation



In [ ]:
reloaded = pd.read_csv(STATION_OUT_PATH)

checks = pd.DataFrame([
    ("Stations written", len(station_output), len(reloaded), len(station_output) == len(reloaded)),
    ("Unique station IDs", station_output["location_id"].nunique(), reloaded["location_id"].nunique(),
     station_output["location_id"].nunique() == reloaded["location_id"].nunique()),
    ("Dusty-hour share within [0, 100]", "yes",
     "yes" if reloaded["dusty_hour_share_pct"].dropna().between(0, 100).all() else "no",
     bool(reloaded["dusty_hour_share_pct"].dropna().between(0, 100).all())),
    ("Assignment offset under 40 km", "yes",
     "yes" if (reloaded["assignment_offset_km"].dropna() < 40).all() else "no",
     bool((reloaded["assignment_offset_km"].dropna() < 40).all())),
    ("Every reconsideration candidate is consensus-flagged", "yes",
     "yes" if reloaded.loc[reloaded["dust_reconsideration_candidate"], "consensus_flagged"].all() else "no",
     bool(reloaded.loc[reloaded["dust_reconsideration_candidate"], "consensus_flagged"].all())),
], columns=["Check", "Computed", "Reloaded", "Pass"]).set_index("Check")

display(table_style(checks))

assert len(station_output) == len(reloaded), "Row count changed on write."
assert reloaded["dusty_hour_share_pct"].dropna().between(0, 100).all(), "Dusty-hour share outside [0, 100]."
assert (reloaded["assignment_offset_km"].dropna() < 40).all(), \
    "A station was assigned to a grid cell further away than this grid's geometry permits."
assert reloaded.loc[reloaded["dust_reconsideration_candidate"], "consensus_flagged"].all(), \
    "A reconsideration candidate is not consensus-flagged -- the two conditions have come apart."


<u>Interpretation</u>

The assignment-offset check uses 40 km rather than the fire cross-reference's ~20 km, since
MERRA-2's coarser grid permits a legitimately larger worst-case offset; a value beyond that
would indicate a coordinate problem rather than normal grid geometry.

## 9. Findings and Limitations

### Findings

1. **A second natural-cause mechanism is now testable alongside fire, not assumed absent.**
<br>Every station carries a dust-exposure figure derived from its own grid cell's hourly
distribution, and every extreme reading carries whether dust was elevated nearby within a
short transport window — the same discipline the fire cross-reference established, extended
to a mechanism fire emissions data cannot see.</br>

2. **DUSMASS25 is used because MERRA-2 already separates dust from other aerosols.**
<br>Unlike total aerosol optical depth, which would require a secondary test such as the
Ångström exponent to distinguish dust from smoke, DUSMASS25 is the reanalysis's own speciated
dust concentration — verified present in the archive in Section 2.2 rather than assumed.</br>

3. **Hourly matching is finer than the fire cross-reference's daily window.** <br>MERRA-2's
native hourly resolution allows episodes to be matched within a few hours rather than a full
day either side, which is a genuine precision improvement over the fire cross-reference where
GFED5's daily granularity was the limiting factor rather than a choice.</br>

4. **This notebook and the fire cross-reference remain independently computed.** <br>Section
7.3 reports their overlap without merging either notebook's verdict into the other, so a
reader can see specifically which mechanism, if either, applies to a given station rather than
a combined score that would obscure the distinction.</br>

### Limitations

1. **The MERRA-2 grid is substantially coarser than GFED5's.** <br>At roughly 0.5° × 0.625°
against GFED5's 0.25° square, cell sharing among stations is more pronounced here, and a dust
coincidence should be read as evidence about a station's general vicinity more than about that
station specifically, more so than the equivalent fire finding.</br>

2. **DUSMASS25 is a model-assimilated concentration, not a direct measurement.** <br>MERRA-2
blends satellite and ground observations into a physical model rather than measuring surface
dust directly at each grid point; its accuracy in a specific remote cell such as inland
Xinjiang has not been independently verified within this notebook.</br>

3. **The dusty-hour threshold embeds a stated convention.** <br>As with the fire threshold,
the 90th-percentile rule and the absolute floor beneath it are reporting conventions, not
derived quantities. Section 5.2 reports the consequences of alternative floors so a reader can
see how much the choice matters.</br>

4. **Dust is one innocent explanation among several, and this notebook only tests one more
of them.** <br>Local construction, agricultural activity, and industrial incidents remain
untested by either this notebook or the fire cross-reference; a station explained by neither
fire nor dust is not thereby confirmed as misreporting.</br>

5. **The acquisition scripts have not been run against the live MERRA-2 service.** <br>Written
against NASA's documented `earthaccess` API and MERRA-2 file specification, but this
environment has no network route to NASA's servers to verify them end to end. The analysis in
this notebook has been tested against synthetic data matching MERRA-2's documented structure;
the first live acquisition run may require minor adjustment.</br>

---

### Output

| File | Contents |
|---|---|
| `DUST_crossref_station.csv` | Per-station dust exposure, reconsideration status |
| `DUST_crossref_episodes.csv` | Per-episode dust coincidence for extreme readings |

**Next:** manual review of stations flagged by neither cross-reference (Section 7.3), and
incorporation of both stations' fire and dust exposure into the report's limitations section.

---

### References

Randles, C. A., et al. (2017). The MERRA-2 Aerosol Reanalysis, 1980 Onward. Part I: System
Description and Data Assimilation Evaluation. *Journal of Climate*, 30(17).

Buchard, V., et al. (2017). The MERRA-2 Aerosol Reanalysis, 1980 Onward. Part II: Evaluation
and Case Studies. *Journal of Climate*, 30(17).
